# 利用卷积神经网络识别石头剪刀布
- 三分类问题

In [59]:
import numpy as np
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
device

device(type='cuda')

## 读取数据
- 读取图片
- train.png 有 21 行，每一行是同一只手的 120 张图片，前 7 行是石头，然后是布和剪刀
- test.png 有 12 行，每一行是同一只手的 31 张图片
- 对图片做预处理
> 1. 拆解图片（2880 * 504 变成 120 * 21 张 24 * 24 的图片）
> 2. 变成灰度图
> 3. 明暗反转（将背景归零，可以让模型忽略背景）

In [60]:
from PIL import Image
image_path = "datasets/guess_gestures/train.png"
image = Image.open(image_path)

width, height = image.size
patch_w = width / 120
patch_h = height / 21

print(f"单张图片尺寸为 : {patch_w, patch_h}")

单张图片尺寸为 : (24.0, 24.0)


- 拆解图片

In [61]:
x_train_np = []
y_train_np = []

for row in range(21):
    if row < 7:
        label = 0# 石头
    elif row < 14:
        label = 1# 步
    else:
        label = 2# 剪刀

    for col in range(120):
        # 计算裁剪的边界，一张一张裁下来
        left = col * patch_w
        right = left + patch_w
        top = row * patch_h
        bottom = top + patch_h

        patch_original = image.crop((left, top, right, bottom))
        patch_grey = patch_original.convert("L")

        # 转换成为np数组，并做归一化
        patch = np.array(patch_grey) / 255.0
        patch = 1 - patch
        x_train_np.append(patch)
        y_train_np.append(label)


x_train_np = np.array(x_train_np)
y_train_np = np.array(y_train_np)
print(f"训练集 : {x_train_np.shape, y_train_np.shape}")

训练集 : ((2520, 24, 24), (2520,))


- 将数据变成tensor

In [62]:
x_train = torch.tensor(x_train_np, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train_np, dtype=torch.long).to(device)

- 增加维度，因为Conv2d接受的参数格式必须是(Batch_Size, Channels, Height, Width)

In [63]:
x_train = x_train.unsqueeze(1)
x_train.shape

torch.Size([2520, 1, 24, 24])

- 封装成Dataset（处理数据的）和DataLoader（加载数据的）

In [64]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(x_train, y_train)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

## 搭建网络

In [65]:
import torch.nn as nn
class MyCNN(nn.Module):
    def __init__(self):
        super(MyCNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()

        self.linear = nn.Linear(in_features=2304, out_features=3)
    def forward(self, _input):
        network = nn.Sequential(
            # 第一层卷积，池化后缩小一半
            self.conv1,
            self.relu,
            self.pool,
            # 第二层卷积，再用池化缩小一半
            self.conv2,
            self.relu,
            self.pool,
            # 第三层卷积
            self.conv3,
            self.relu,
            self.flatten,
            self.linear,
        )
        return network(_input)

- 实例化模型，并且放到`device`上

In [66]:
my_cnn_classifier = MyCNN()
my_cnn_classifier.to(device)

MyCNN(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu): ReLU()
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear): Linear(in_features=2304, out_features=3, bias=True)
)

- 声明损失函数和优化器

In [67]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(my_cnn_classifier.parameters(), lr=0.001)

## 训练

In [68]:
# 声明训练轮数 (Epochs)
epochs = 20

# 开启训练模式 (启用 Dropout 和 BatchNorm 等的训练特性)
my_cnn_classifier.train()

for epoch in range(epochs):
    running_loss = 0.0

    # 遍历 DataLoader 里的每一个 Batch
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        # 1. 梯度清零 (Zero gradients)
        optimizer.zero_grad()

        # 2. 前向传播 (Forward)
        outputs = my_cnn_classifier(inputs)

        # 3. 计算损失 (Loss)
        loss = loss_fn(outputs, labels)

        # 4. 反向传播 (Backward)
        loss.backward()

        # 5. 梯度下降 (Optimizer step)
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(dataloader):.4f}")

Epoch [1/20], Loss: 0.8951
Epoch [2/20], Loss: 0.3531
Epoch [3/20], Loss: 0.1117
Epoch [4/20], Loss: 0.0503
Epoch [5/20], Loss: 0.0214
Epoch [6/20], Loss: 0.0151
Epoch [7/20], Loss: 0.0086
Epoch [8/20], Loss: 0.0055
Epoch [9/20], Loss: 0.0033
Epoch [10/20], Loss: 0.0024
Epoch [11/20], Loss: 0.0019
Epoch [12/20], Loss: 0.0015
Epoch [13/20], Loss: 0.0013
Epoch [14/20], Loss: 0.0010
Epoch [15/20], Loss: 0.0009
Epoch [16/20], Loss: 0.0007
Epoch [17/20], Loss: 0.0006
Epoch [18/20], Loss: 0.0005
Epoch [19/20], Loss: 0.0004
Epoch [20/20], Loss: 0.0004


## 预测

In [69]:
# 1. 切换到评估模式 (非常重要)
my_cnn_classifier.eval()

def predict_single_gesture(image_path, model, device):
    # 读取并强制转换为灰度图
    img = Image.open(image_path).convert('L')

    # 强制缩放到训练时的尺寸 24x24
    img = img.resize((24, 24))

    # 转为 numpy 数组并归一化到 0-1
    img_array = np.array(img) / 255.0

    # 【对齐训练集】如果你在训练时做了明暗反转（黑底白手），这里必须保持一致！
    if img_array[0, 0] > 0.5:
        img_array = 1.0 - img_array

    # 将 numpy 数组转为 Tensor
    # 当前 img_array 维度是 (24, 24)
    # 我们需要将其扩展为 (Batch=1, Channel=1, Height=24, Width=24)
    input_tensor = torch.tensor(img_array, dtype=torch.float32)
    input_tensor = input_tensor.unsqueeze(0).unsqueeze(0).to(device)

    # 2. 关闭梯度计算进行推理
    with torch.no_grad():
        # 前向传播
        output = model(input_tensor)
        # output 的长相类似 tensor([[-1.2, 5.3, 0.1]])

        # 提取最大概率的索引 (0:石头, 1:布, 2:剪刀)
        predicted_idx = torch.argmax(output, dim=1).item()

    # 标签映射词典
    label_map = {0: '石头 (Rock)', 1: '布 (Paper)', 2: '剪刀 (Scissors)'}

    return label_map[predicted_idx]

# 假设你有一张名为 test_hand.png 的图片
# prediction = predict_single_gesture("test_hand.png", my_cnn_classifier, device)
# print(f"预测结果: {prediction}")

In [70]:
# 读取测试集大图
test_image = Image.open("datasets/guess_gestures/test.png")
test_w, test_h = test_image.size

# 计算测试集单张小图的尺寸
t_patch_w = int(test_w / 31)
t_patch_h = int(test_h / 12)

correct_count = 0
total_count = 12 * 31

# 这里假设 test.png 的排版也是：前4行石头(0)，中4行布(1)，后4行剪刀(2)
# 如果标签分布不同，请修改下方的 if-elif 逻辑
for row in range(12):
    if row < 4:
        true_label = 0
    elif row < 8:
        true_label = 1
    else:
        true_label = 2

    for col in range(31):
        left = col * t_patch_w
        upper = row * t_patch_h
        right = left + t_patch_w
        lower = upper + t_patch_h

        # 裁剪并灰度化
        patch = test_image.crop((left, upper, right, lower)).convert('L')
        # 确保缩放到 24x24
        patch = patch.resize((24, 24))

        img_array = np.array(patch) / 255.0
        if img_array[0, 0] > 0.5:
            img_array = 1.0 - img_array

        # 增加 Batch 和 Channel 维度：(1, 1, 24, 24)
        input_tensor = torch.tensor(img_array, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)

        with torch.no_grad():
            output = my_cnn_classifier(input_tensor)
            predicted_idx = torch.argmax(output, dim=1).item()

            if predicted_idx == true_label:
                correct_count += 1

print(f"测试集总数量: {total_count}")
print(f"预测正确数量: {correct_count}")
print(f"模型准确率: {correct_count / total_count * 100:.2f}%")

测试集总数量: 372
预测正确数量: 328
模型准确率: 88.17%
